# Fine-tuning Hugging Face sequence classifiers on the English-translated tweets

In [ ]:
DATA_DIR = "../data"

In [3]:
!pip install transformers -q
!pip install datasets -q
!pip install accelerate -U -q

In [4]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [9]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
train_df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 4


In [ ]:
import re

def remove_noise(text):
    text = str(text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove mentions
    text = re.sub(r'(\s)?@\S+', r'\1', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE [^\]]+\]', '', text)
    # remove whitespace leading & trailing
    text = text.strip("RT ")
    text = text.strip()
    return text

def drop_duplicated(df):
    print("Df size:", train_df.shape)
    train_df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", train_df.shape)
    train_df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", train_df.shape)

In [ ]:
train_df['text'] = train_df['text'].apply(remove_noise)
drop_duplicated(train_df)
train_df.head()

Df size: (5000, 2)
After dropping literal duplicated rows, df size:  (4340, 2)
After dropping text duplicated rows, df size:  (4218, 2)


,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,"emng bener sih, pendukung 01 ada yg goblok, be...",Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap ASN termasuk TNI dan Polr...,Politik


In [5]:
def remove_noise_eng(text):
    text = str(text)
    text = text.replace("&amp;", "and")
    text = text.replace("​", "")
    text = text.strip()
    return text

In [6]:
train_df = pd.read_csv(f"{DATA_DIR}/train_eng_v2.csv")
train_df["text"] = train_df["text"].apply(remove_noise_eng)
train_df.head()

                                                text             label
0  Prabowo's visit was to inaugurate and hand ove...  Sumber Daya Alam
1  Anies received a standing ovation when he beca...           Politik
2  It's true, there are supporters of 01 who are ...         Demografi
3  When Anies was critical of Pak Prabowo's perfo...           Politik
4  Anies Baswedan hopes that ASN, including the T...           Politik

In [7]:
label_to_id = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}
id_to_label = {v: k for k, v in label_to_id.items()}
label_to_id

{'Politik': 0,
 'Sosial Budaya': 1,
 'Ekonomi': 2,
 'Pertahanan dan Keamanan': 3,
 'Ideologi': 4,
 'Sumber Daya Alam': 5,
 'Demografi': 6,
 'Geografi': 7}

In [8]:
# Split the DataFrame into training and validation sets
train_df_split, val_df_split = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)

train_df_split["label"] = train_df_split["label"].map(label_to_id)
val_df_split["label"] = val_df_split["label"].map(label_to_id)

# train_df_split["label"] = train_df_split["label"].map(lambda x : np.eye(8)[x])
# val_df_split["label"] = val_df_split["label"].map(lambda x : np.eye(8)[x])

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df_split)
val_dataset = Dataset.from_pandas(val_df_split)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [9]:
train_df_split["label"].to_numpy()

array([0, 0, 0, ..., 0, 0, 0])

In [10]:
# MODEL = 'StevenLimcorn/indonesian-roberta-base-emotion-classifier'
# MODEL = 'w11wo/indonesian-roberta-base-sentiment-classifier'
# MODEL = 'ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa'
# MODEL = 'cardiffnlp/tweet-topic-latest-multi'
# MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
# MODEL = 'FacebookAI/roberta-base'
# MODEL = 'FacebookAI/roberta-large-mnli'
# MODEL = 'j-hartmann/emotion-english-distilroberta-base'
MODEL = 'yiyanghkust/finbert-tone'

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

Map:   0%|          | 0/3374 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

In [12]:
# Load the model
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=8, ignore_mismatched_sizes=True)

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at yiyanghkust/finbert-tone and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir=f"./{MODEL}_results_ori",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)


# Train the model
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


In [1]:
# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

NameError: name 'trainer' is not defined

In [ ]:
rm -rf cardiffnlp/*

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
